# COMP47670 Assignment 1 - Autumn 2026/27
## Task 1: Data Collection

**Student Name**: Ellen Lyons

**Student Number**: 26117753

In this first task, I will use web-scraping techniques to collect and parse data from the data source '*Property Sale Register*'. 

First I had a look at the ``robots.txt`` file for the website http://mlg.ucd.ie/robots.txt and here is what it said:

```User-agent: *
disallow: /files/*

User-agent: *
disallow: /datasets/comp30120/*

User-agent: *
disallow: /datasets/comp41450/*

User-agent: *
disallow: /datasets/comp41680/*
```

Ok so I can't scrape any paths that start with ```/files/``` or ```/datasets/``` but the website I am scraping is  ```http://mlg.ucd.ie/modules/python/assign1/property/index.html```, so we are all good. 

(Here is where I got my explanation of robots files: https://medium.com/@mikram2015/understanding-robots-txt-the-beginners-guide-for-web-scrapers-e4643d4e8f38)

I am going to use the *Beautiful Soup* package to extract HTML info about the property sales.

In [1]:
import bs4 
import urllib.request

Throughout this assignment, I will often use statements like ``try``/``except`` or ``if``/``else`` so that if there is an error retrieivng data, or an error within a file, my selected warning will be printed rather than Python printing me an error message. 

This is just good practise, and again, I am following this from the lectures/example code. 

In [2]:
# property sales website 
home_url = "http://mlg.ucd.ie/modules/python/assign1/property/2022-page01.html"

try:
    response = urllib.request.urlopen(home_url)
    html = response.read().decode("utf-8")
    print(html)
except Exception as e:
    print(f"Error retrieving page: {e}")

<!DOCTYPE html>
<html lang="en">
<head>
  <meta charset="utf-8">
  <meta name="robots" content="noindex">  
  <meta name="description" content="Property Sale Register 2022 - Detailed property transaction records for the year 2022">
  <meta http-equiv="X-UA-Compatible" content="IE=edge">
  <meta name="viewport" content="width=device-width, initial-scale=1">
  <title>Property Sale Register - Sales 2022</title>
  <link rel="stylesheet" href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css">
  <link rel="stylesheet" type="text/css" href="style.css">
</head>
<body>
  <div class="container">
    <main>
      <div class="page-header" style="margin-bottom: 0;">
        <h1><a href="index.html">Property Register</a> &mdash; 2022 Sales</h1>
      </div>
      
      <div class="total">
        Page 1 of 17 &mdash;
        Displaying results 1 to 20 from total of 323 records for this year
      </div>

      <section class="property-listings">
        <ol start="1">
        

I clicked around the website I chose to see how it is formatted. So the homepage leads to four more links, labelled by the year, 2022, 2023, 2024 and 2025. Each year then contains x amount of pages of data that tell us about different properties sold that year. So I need to make a loop that goes over the four years, and another loop that scrapes the data from each page within each year. 

In [8]:
# create a BeautifulSoup object which contains the structure of the HTML page
home = bs4.BeautifulSoup(html, 'html.parser')

In [12]:
# each year has the base + the path of the individual year
base = "http://mlg.ucd.ie/modules/python/assign1/property/"

# loop through the hrefs to get each years first page
years = []

for a in home.find_all("a"):
    years.append(base + a.get("href"))
    
years

['http://mlg.ucd.ie/modules/python/assign1/property/2022-page01.html',
 'http://mlg.ucd.ie/modules/python/assign1/property/2023-page01.html',
 'http://mlg.ucd.ie/modules/python/assign1/property/2024-page01.html',
 'http://mlg.ucd.ie/modules/python/assign1/property/2025-page01.html']

In [8]:
url_2022 = "http://mlg.ucd.ie/modules/python/assign1/property/2022-page01.html"
response = urllib.request.urlopen(url_2022)
html = response.read().decode("utf-8")
soup = bs4.BeautifulSoup(html, 'html.parser')
print(soup.prettify())

<!DOCTYPE html>
<html lang="en">
 <head>
  <meta charset="utf-8"/>
  <meta content="noindex" name="robots"/>
  <meta content="Property Sale Register 2022 - Detailed property transaction records for the year 2022" name="description"/>
  <meta content="IE=edge" http-equiv="X-UA-Compatible"/>
  <meta content="width=device-width, initial-scale=1" name="viewport"/>
  <title>
   Property Sale Register - Sales 2022
  </title>
  <link href="https://maxcdn.bootstrapcdn.com/bootstrap/3.3.5/css/bootstrap.min.css" rel="stylesheet"/>
  <link href="style.css" rel="stylesheet" type="text/css"/>
 </head>
 <body>
  <div class="container">
   <main>
    <div class="page-header" style="margin-bottom: 0;">
     <h1>
      <a href="index.html">
       Property Register
      </a>
      — 2022 Sales
     </h1>
    </div>
    <div class="total">
     Page 1 of 17 —
        Displaying results 1 to 20 from total of 323 records for this year
    </div>
    <section class="property-listings">
     <ol start="1">

What I have done below is parse the data for page 1 of the 2022 property sales. I loop over all sales on the page, then loop over each table and extract the info. I got a bit stuck on looping over the rows to add them to th edictionary, but I used the logic from 
```
# Source - https://stackoverflow.com/a/51657193
# Posted by Andrej Kesely
# Retrieved 2026-10-21, License - CC BY-SA 4.0
```


In [66]:
property_sales = [] # list to store results

sale_matches = soup.find_all("li") # find all li tags we want

if not sale_matches: # check if there are any results
    print("No property sale records found")

else:
    print(f"Processing {len(sale_matches)} house sale records ...")
    
    for i, match in enumerate(sale_matches, 1):
        # within each property, find span tag and extract text from each one
        date_sold_elem = match.find("span", {"class": "sold"}) # there is only one span tag!
        
        # check all required elements are found
        if date_sold_elem is None:
            print(f"Warning: Property sale record {i} is missing required fields")
            continue
        
        # extract text 
        date_sold = date_sold_elem.text.strip()
        
        # add info to a dictionary
        property_sale = {"Date Sold": date_sold}
        
        
        table = match.find("table", {"class": "sale"})

        if table is None:
            print(f"No further info about property {i} found")
            continue
        #print(table.prettify())
        
        rows = table.find_all("tr") # tr stands for table row 
        for row in rows:
            cells = row.find_all("td") # td stands for table data cell
            if len(cells) == 2: # there should only be two 'td' tags
                key = cells[0].text.strip()
                value = cells[1].text.strip()
                #print(key, value)
                property_sale[key] = value
            else:
                print("Issue with number of data cells")
                
        
        property_sales.append(property_sale)
                
print(f"Successfully extracted {len(property_sales)} complete property sale records!")

Processing 20 house sale records ...
Successfully extracted 20 complete property sale records!


In [67]:
property_sales

[{'Date Sold': 'Sold 26 December 2022',
  'Sale Price:': '€359,086.00',
  'Location:': 'Oak Park',
  'Year Built:': '1984',
  'Size:': '1,131sqft',
  'Garden:': 'Yes',
  'Garage:': 'Yes',
  'Property Details:': 'Type: Detached; Style: 1.5-Storey; 3 Bedrooms; 1 Bathroom',
  'First Time Buyer:': 'Yes',
  'Property Reference:': 'PROP-105204'},
 {'Date Sold': 'Sold 2022-12-25',
  'Sale Price:': '€734,419.00',
  'Location:': 'Porter Square',
  'Year Built:': '2019',
  'Size:': '1,941sqft',
  'Garden:': 'Yes',
  'Garage:': 'Yes',
  'Property Details:': 'Type: Detached; 2 Bathrooms; Style: 2-Storey; 3 Bedrooms',
  'First Time Buyer:': 'NO',
  'Property Reference:': 'PROP-105203'},
 {'Date Sold': 'Sold 2022-12-19',
  'Sale Price:': '€724,835.00',
  'Location:': 'Porter Square',
  'Year Built:': '2021',
  'Size:': '1,996 sqft',
  'Garden:': 'Yes',
  'Garage:': 'Yes',
  'Property Details:': 'Type: Det.; Style: 2-Storey; 3 Bedrooms; 2 Bathrooms',
  'First Time Buyer:': 'No',
  'Property Reference